# Observing Phenometrics in Madagascar using HLS
---
**Summary**
This tutorial demonstrates how to use Harmonized Landsat and Sentinel-2 (HLS) 10-day composite images to extract NDVI, EVI, and EVI2 values. It also demonstrates how to use MERRA-2 data to obtain real climate information to compare with the HLS values.

**What is HLS data?**
The Harmonized Landsat and Sentinel-2 (HLS) project combines data from NASA/USGS Landsat 8 and Landsat 9 satellites and the European Space Agency's Sentinel-2A, Sentinel-2B, and Sentinel-2C satellites. The project makes it so land surface observations can be obtained at a 30-meter resolution every 1.6 days. This notebook uses HLS composite images of 10-day periods from 2016-2025.

Phenometrics: phenological (cyclic and natural phenomena) transition dates

## Tutorial Outline

1. Environment Setup
2. NDVI, EVI, EVI2 Calculation
3. Initialize variables and grab data from S3
4. MODIS and HLS comparison

## 1. Environment Setup
**Environment:** conda env create -f global-airborne-core-20251107.yaml

In [1]:
# import packages
import os
os.environ['AWS_NO_SIGN_REQUEST'] = 'YES' 

import re
import gc
from pathlib import Path
from datetime import datetime, timedelta
import rioxarray as rxr
import xarray as xr
import rasterio
import geopandas as gpd
import numpy as np
import folium
import pandas as pd
import h5py
import math

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib import dates, colors
from matplotlib.colors import Normalize
from rasterio.transform import array_bounds

from shapely.geometry import box
from shapely.geometry import Point
from shapely.geometry import shape
import glob
from scipy import stats

import boto3
from botocore import UNSIGNED
from botocore.client import Config

In [2]:
from matplotlib import colors
from matplotlib.colors import ListedColormap

## 2. NDVI, EVI, EVI2 calculation
From the HLS 10-day composites, this code creates .tif files for the NDVI, EVI, and EVI2 vegetation indices. 

**NDVI**: Normalized Difference Vegetation Index
- Index used to quantify the health and density of vegetation using satellite imagery
- Values fall between -1 to 1
- A barren area has NDVI = 0. An area with dense vegetation has NDVI = 1. An ocean has NDVI = -1.
- $\text{NDVI} = \frac{NIR - Red}{NIR + Red}$

**EVI**: Enhanced Vegetation Index
- More sensitive in areas with more dense vegetation, making it better for this region in Madagascar
- Reduces atmospheric influence and minimizes soil background effects
- $\text{EVI} = G \times \frac{NIR - Red}{NIR + C1 \times Red - C2 \times Blue + L}$
  - G: gain factor
  - NIR, Red, Blue: Near-Infrared, Red, and Blue surface reflectance bands
  - C1, C2: coefficients of aerosol resistance
  - L: canopy background adjustment
  - In this notebook, we use G = 2.5, C1 = 6, C2 = 7.5, and L = 1, which are the coefficients for the MODIS-EVI algorithm

**EVI2**: 2-band EVI
- Allows the EVI calculation to extend farther back in time, as the Blue band does not exist on older AVHRR sensors
- Sensor level atmospheric adjustment has improved since the initial inclusion of the Blue band (which was primarily used to mitigate aerosol interference), making the impact of Blue minimal.
- $\text{EVI2} = G \times \frac{NIR - Red}{L + NIR + C \times R}$
  - G, L, and C are found to minimize the difference between EVI and EVI2
  - In this notebook, we use G = 2.5, L = 2.4, and C = 1

In [ ]:
def ndvi_calc(nir, red):
     ndvi = (nir - red)/(nir + red)
     lower_bound = ndvi >= -2
     upper_bound = ndvi <= 10
     valid = lower_bound & upper_bound
     return ndvi.where(valid)

def evi_calc(red, nir, blue):
    evi = (2.5 * (nir - red)) / ((nir + 6 * red - 7.5 * blue) + 1)
    lower_bound = evi >= -2
    upper_bound = evi <= 10
    valid = lower_bound & upper_bound
    return evi.where(valid)

def evi2_calc(red, nir):
    evi2 = 2.5 * (nir - red) / (nir + 2.4 * red + 1) # Jiang et al., 2008 
    lower_bound = evi2 >= -2
    upper_bound = evi2 <= 10
    valid = lower_bound & upper_bound
    return evi2.where(valid)

QA_BIT = {
    "cirrus": 0,
    "cloud": 1,
    "adj_cloud": 2,
    "cloud shadow": 3,
    "snowice": 4,
    "water": 5,
    "aerosol_l": 6,
    "aerosol_h": 7,
}

## 3. Initialize variables and grab data from S3
The HLS composite files are stored in S3 storage. This code grabs the composites and creates the NDVI, EVI, and EVI2 `.tif` files to be stored locally.

In [3]:
tile_id = "39KUA" # Madagascar tiles: 39LUD, 39LVD, 39LUC, 39LVC
bucket_name = 'hls-composite-bdec' # S3 bucket name
veg_index = "ndvi" #"evi", "evi2"

prefix = f'step-2-evaluation-prototype-data/composite-median-10day/{tile_id}/' # file prefix where data is stored in S3
start_year = 2016
end_year = 2025
year_dirs = [f"{prefix}{year}" for year in range(start_year, end_year + 1)]

output_dir = f"hls_veg_indices/{tile_id}"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
year_dirs

### Grab composites and create `.tif` files for vegetation indices
The below function:
1. Grabs 10-day HLS composite files for the designated tile from S3 storage
2. Uses surface reflectance bands to calculate the designated vegetation index for each 10-day composite, and stores it locally in a `.tif` file 

In [ ]:
def make_url(key):
    return f"/vsicurl/https://{bucket.name}.s3.amazonaws.com/{key}"
    
def mask_and_scale(da, fmask):
    bad_pixel_mask = (
        ((fmask & (1 << QA_BIT["cloud"])) > 0)
        | ((fmask & (1 << QA_BIT["adj_cloud"])) > 0)
        | ((fmask & (1 << QA_BIT["cloud shadow"])) > 0)
        | (fmask == -9999)
        | ((da < 0))
        | ((fmask & (1 << QA_BIT["water"])) > 0)
        | ((fmask & (1 << QA_BIT["snowice"])) > 0)
    )                              
    scaled_da = da.where(da != -9999) / 10000 
    masked_da = scaled_da.where(~bad_pixel_mask)
    return masked_da
                
def get_files_from_s3():
    collected_dates = []

    for year_prefix in year_dirs:
        print(f"\nProcessing year prefix: {year_prefix}")
        print(f'{bucket.name}/{year_prefix}')

        response = bucket.meta.client.list_objects_v2(
            Bucket=bucket.name,
            Prefix=year_prefix + "/",
            Delimiter="/"
        )

        cadence_prefixes = [cp["Prefix"] for cp in response.get("CommonPrefixes", [])]
        print(cadence_prefixes[0])

        date_pattern = r'HLS\.M30\.T[A-Z0-9]+\.(\d{7})\.(\d{7})\.\d+\.\d+'

        for cadence_prefix in sorted(cadence_prefixes):
            cadence_name = cadence_prefix.rstrip("/").split("/")[-1]
            print(cadence_name)

            match = re.search(date_pattern, cadence_name)
            if not match:
                print(f"Warning: Date pattern not found in {cadence_name}")
                continue

            start_date_str = match.groups()[0]
            try:
                year_from_date = int(start_date_str[:4])
                day_of_year    = int(start_date_str[4:])
                date_obj       = datetime(year_from_date, 1, 1) + timedelta(days=day_of_year - 1)
            except Exception:
                print(f"Warning: Could not parse date from {cadence_name}")
                continue

            red_file = nir_file = blue_file = doy_file = fmask_file = None

            for obj in bucket.objects.filter(Prefix=cadence_prefix):
                if obj.key.endswith(".Red.tif"):
                    red_file = obj.key
                elif obj.key.endswith(".NIR_Narrow.tif"):
                    nir_file = obj.key
                elif obj.key.endswith(".Blue.tif"):
                    blue_file = obj.key
                elif obj.key.endswith(".DOY.tif"):
                    doy_file = obj.key
                elif obj.key.endswith(".Fmask.tif"):
                    fmask_file = obj.key

            if red_file is None or nir_file is None or fmask_file is None:
                print(f"Missing Red, NIR, or Fmask - skipping {cadence_name}")
                continue

            fmask = rxr.open_rasterio(make_url(fmask_file), masked=True).squeeze("band", drop=True).astype(np.int8)
            red   = rxr.open_rasterio(make_url(red_file),   masked=True).squeeze("band", drop=True)
            nir   = rxr.open_rasterio(make_url(nir_file),   masked=True).squeeze("band", drop=True)
            red_masked = mask_and_scale(red, fmask)
            nir_masked = mask_and_scale(nir, fmask)
            
            for veg_index in ["evi", "evi2", "ndvi"]:                
                if veg_index == 'evi2':
                    vi         = evi2_calc(red_masked, nir_masked)
                    out_suffix = "EVI2"
    
                elif veg_index == 'evi':
                    if veg_index == 'evi' and blue_file is None:
                        print(f"Missing Blue for EVI - skipping {cadence_name}")
                        continue
                    blue        = rxr.open_rasterio(make_url(blue_file), masked=True).squeeze("band", drop=True)
                    blue_masked = mask_and_scale(blue, fmask)
                    vi          = evi_calc(red_masked, nir_masked, blue_masked)
                    out_suffix  = "EVI"
                    del blue, blue_masked
    
                elif veg_index == 'ndvi':
                    vi         = ndvi_calc(nir_masked, red_masked)
                    out_suffix = "NDVI"
    
                else:
                    print(f"Unknown veg_index '{veg_index}' - skipping")
                    del red, nir, red_masked, nir_masked, fmask
                    gc.collect()
                    continue
    
                vi = vi.expand_dims("time").assign_coords(time=[date_obj])
                collected_dates.append(date_obj)
    
                doy_str     = date_obj.strftime('%Y%j')
                output_file = f"{output_dir}/{tile_id}_10day_median_{doy_str}_{out_suffix}.tif"
                vi.attrs.pop('scale_factor', None)
                vi.attrs.pop('add_offset',   None)
                vi.rio.to_raster(output_file, compress="lzw")
                print(f"Written: {output_file}")

            if doy_file:
                doy_output = f"{output_dir}/{tile_id}_DOY_10day_{doy_str}.tif"
                bucket.meta.client.download_file(bucket.name, doy_file, doy_output)
                print(f"Copied DOY: {doy_output}")
            else:
                print(f"Warning: No DOY tif found for {cadence_name}")

            del red, nir, red_masked, nir_masked, fmask, vi
            gc.collect()

    print(f"Saved {len(collected_dates)} {veg_index.upper()} composites")

In [ ]:
s3 = boto3.resource('s3', config=Config(signature_version=UNSIGNED))
bucket = s3.Bucket(bucket_name)

In [ ]:
get_files_from_s3()

*** NOTE: with this setup, if don't alreay have data, have to run individually for each veg index

probably would be smart to split this into 2 notebooks -- one for downloading data and one for analysis